# 1장. 개발 환경과 프로젝트 구조

| 항목 | 내용 |
|---|---|
| 예상 시간 | 2시간 |
| 선수 장 | 0장 |
| 이 장의 결과물 | 모든 라이브러리가 동작하는 환경, 과제용 폴더 구조, `requirements.txt` |

**학습 목표**
- 가상환경이 왜 필요한지 설명하고 uv로 패키지를 추가하고 실행할 수 있다.
- `python 파일.py`와 `python -m 모듈`의 차이를 알고 `ModuleNotFoundError`를 해결할 수 있다.
- KoNLPy가 Java를 필요로 하는 이유를 알고 설치를 확인할 수 있다.
- `requirements.txt`와 `.gitignore`에 무엇을 넣어야 하는지 안다.

> 이 장은 터미널 작업이 많습니다. `bash`라고 표시된 회색 블록은 **노트북이 아니라 터미널에서** 실행하는 명령입니다. 파이썬 셀은 노트북에서 실행해 결과를 확인합니다.

## 1.1 가상환경과 uv

### 가상환경이 필요한 이유
A 과제는 `numpy 1.x`, B 과제는 `numpy 2.x`가 필요하다면, 컴퓨터에 파이썬 하나만 있을 때 둘 중 하나는 깨집니다. **가상환경**은 프로젝트마다 독립된 파이썬과 라이브러리 묶음을 만들어 이 충돌을 막습니다.

### uv
`uv`는 파이썬 버전, 가상환경, 패키지 설치를 한 도구로 관리합니다. 설치는 한 번만 하면 됩니다.

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh
# 또는 Homebrew
brew install uv
```

| 명령 | 하는 일 |
|---|---|
| `uv init` | 새 프로젝트 만들기 (`pyproject.toml` 생성) |
| `uv add numpy` | 패키지 추가 (`pyproject.toml`과 `uv.lock`에 기록하고 설치) |
| `uv remove numpy` | 패키지 제거 |
| `uv sync` | `uv.lock`에 적힌 그대로 환경 맞추기 (다른 컴퓨터에서 처음 받을 때) |
| `uv run python main.py` | 가상환경 안의 파이썬으로 실행 |
| `uv run jupyter lab` | 가상환경 안에서 JupyterLab 실행 |

### 이 저장소의 규칙
- 저장소 **루트에 환경이 하나** 있고(`pyproject.toml`, `uv.lock`, `.venv/`), 모든 과제 폴더가 이 환경을 함께 씁니다.
- 과제 폴더 안에서 `uv run ...`을 실행해도 uv가 위쪽 폴더의 `pyproject.toml`을 찾아 루트 환경을 씁니다.
- `pip install`을 직접 쓰지 않습니다. pip로 설치하면 `uv.lock`에 기록되지 않아 다른 사람이 같은 환경을 재현할 수 없습니다.

```bash
cd projects/a4-1-document-search
uv run python main.py --mode search --query "space shuttle" --topk 5
```

지금 이 노트북이 어떤 파이썬으로 실행되고 있는지 확인해 봅시다.

In [1]:
import sys
from pathlib import Path

print("파이썬 버전:", sys.version.split()[0])
# 경로 끝부분이 .venv/bin/python(Windows는 .venv\Scripts\python.exe)이면 가상환경입니다.
print("실행 파일  :", Path(*Path(sys.executable).parts[-3:]))
print("3.10 이상? :", sys.version_info >= (3, 10))

파이썬 버전: 3.13.5
실행 파일  : .venv/bin/python
3.10 이상? : True


이 과제에 필요한 라이브러리가 설치되어 있는지, 버전은 무엇인지 확인합니다. `importlib.metadata.version()`은 설치된 패키지의 버전을 알려 줍니다.

In [2]:
from importlib.metadata import PackageNotFoundError, version

# (설치 이름, 이 과제에서 쓰는 곳)
PACKAGES = [
    ("numpy", "TF-IDF, 코사인 유사도 직접 구현"),
    ("pandas", "데이터 로드와 정리"),
    ("scikit-learn", "TF-IDF 검증, 분류 모델, 평가 지표"),
    ("matplotlib", "그래프, 혼동 행렬 그림"),
    ("seaborn", "히트맵 (선택)"),
    ("nltk", "영어 불용어, 토큰화"),
    ("konlpy", "한국어 형태소 분석 (Okt)"),
    ("pytest", "테스트"),
]

for name, usage in PACKAGES:
    try:
        ver = version(name)
    except PackageNotFoundError:
        ver = "❌ 설치 안 됨"
    print(f"{name:<13} {ver:<12} {usage}")

numpy         2.5.3        TF-IDF, 코사인 유사도 직접 구현
pandas        3.0.6        데이터 로드와 정리
scikit-learn  1.9.1        TF-IDF 검증, 분류 모델, 평가 지표
matplotlib    3.11.2       그래프, 혼동 행렬 그림
seaborn       0.13.2       히트맵 (선택)
nltk          3.10.3       영어 불용어, 토큰화
konlpy        0.6.0        한국어 형태소 분석 (Okt)
pytest        9.1.1        테스트


## 1.2 폴더 구조

과제 코드는 역할에 따라 폴더를 나눕니다.

```
a4-1-document-search/
├── main.py             # 실행 진입점 (CLI). 다른 모듈을 불러서 조립만 한다
├── requirements.txt    # 제출 필수: 필요한 라이브러리와 버전
├── README.md           # 분석 리포트 (구현 요약 → 검증/실험 → 한계)
├── src/                # 핵심 코드. 노트북과 main.py가 여기서 import한다
│   ├── __init__.py     #   이 폴더를 "패키지"로 만드는 파일 (비어 있어도 됨)
│   ├── data.py         #   데이터 불러오기
│   ├── preprocess.py   #   정제, 토큰화, 불용어
│   ├── tfidf.py        #   NumPy TF-IDF
│   ├── verify.py       #   sklearn과 비교 검증
│   ├── search.py       #   코사인 유사도 검색
│   └── classify.py     #   분류와 평가
├── tests/              # pytest 테스트
├── data/               # 원본 데이터와 캐시 (git에 올리지 않음)
├── outputs/            # 실행 결과 (검증 로그, 혼동 행렬 그림 등)
├── notebooks/          # 탐색용 노트북
└── studys/             # 이 학습 자료
```

| 어디에 | 무엇을 | 이유 |
|---|---|---|
| 노트북 | 실험, 시각화, 결과 확인 | 셀 단위로 바로 실행해 보며 탐색하기 좋다 |
| `src/*.py` | 재사용하는 함수와 클래스 | 노트북, `main.py`, 테스트가 **같은 코드**를 불러 쓴다 |
| `tests/` | 함수가 맞게 동작하는지 확인하는 코드 | 코드를 고친 뒤에도 결과가 깨지지 않았는지 자동으로 확인 |

> 노트북에서만 구현하면 `main.py`와 테스트에서 그 코드를 쓸 수 없습니다. **실험은 노트북에서, 확정된 코드는 `src/`로** 옮기는 습관을 들이세요.

이 과제 폴더의 실제 구조를 확인해 봅시다.

In [3]:
PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
print(PROJECT.name + "/")
for p in sorted(PROJECT.iterdir()):
    if p.name.startswith("."):
        continue
    print("  " + p.name + ("/" if p.is_dir() else ""))

a4-1-document-search/
  README.md
  data/
  main.py
  notebooks/
  outputs/
  reports/
  requirements.txt
  src/
  studys/
  tests/


## 1.3 모듈, 패키지, import

- **모듈**: `.py` 파일 하나. `preprocess.py`는 `preprocess` 모듈입니다.
- **패키지**: 모듈을 담은 폴더. `src/`는 `src` 패키지이고, `from src.preprocess import clean`처럼 씁니다.

`import`가 모듈을 찾는 위치 목록이 `sys.path`입니다. 여기서 초보자가 가장 자주 만나는 에러가 생깁니다.

| 실행 방법 | `sys.path`의 첫 번째 위치 | `from src.x import ...` |
|---|---|---|
| `python src/verify.py` | 실행한 **파일이 있는 폴더** (`src/`) | ❌ `src/` 안에서 `src`를 찾으니 실패 |
| `python -m src.verify` | **현재 폴더** (프로젝트 루트) | ✅ 루트에 `src/`가 있으니 성공 |

직접 확인해 봅시다. 임시 폴더에 작은 패키지를 만들고 두 방법으로 실행합니다.

In [4]:
import os
import subprocess
import tempfile
import textwrap

tmp = Path(tempfile.mkdtemp())
pkg = tmp / "mypkg"
pkg.mkdir()
(pkg / "__init__.py").write_text("")
(pkg / "textutil.py").write_text(textwrap.dedent("""
    def shout(text: str) -> str:
        return text.upper() + "!"
"""))
(pkg / "run.py").write_text(textwrap.dedent("""
    from mypkg.textutil import shout

    if __name__ == "__main__":
        print(shout("hello"))
"""))


def run(args):
    """임시 폴더를 현재 폴더로 삼아 파이썬을 실행하고 마지막 출력 줄을 보여 준다."""
    env = {**os.environ, "PYTHON_COLORS": "0"}  # 에러 메시지의 터미널 색상 코드 끄기
    result = subprocess.run([sys.executable, *args], cwd=tmp, capture_output=True, text=True, env=env)
    lines = (result.stdout + result.stderr).strip().splitlines()
    print(f"$ python {' '.join(args)}")
    print(f"  → {lines[-1] if lines else ''}\n")


run(["mypkg/run.py"])     # 파일 경로로 실행
run(["-m", "mypkg.run"])  # 모듈 이름으로 실행

$ python mypkg/run.py
  → ModuleNotFoundError: No module named 'mypkg'

$ python -m mypkg.run
  → HELLO!



파일 경로로 실행하면 `ModuleNotFoundError`, `-m`으로 실행하면 정상입니다. 그래서 이 과제는 **프로젝트 폴더에서** `uv run python -m src.verify`처럼 실행합니다. `main.py`는 프로젝트 루트에 있으므로 `python main.py`로 실행해도 `src`를 찾을 수 있습니다.

### `if __name__ == "__main__":`
파일을 **직접 실행**하면 `__name__`이 `"__main__"`이 되고, 다른 곳에서 **import**하면 모듈 이름이 됩니다. 이 조건문 안의 코드는 직접 실행할 때만 돌아가므로, 같은 파일을 실행 스크립트로도, 라이브러리로도 쓸 수 있습니다.

In [5]:
sys.path.insert(0, str(tmp))  # 노트북에서 임시 패키지를 import할 수 있게 경로 추가
import mypkg.run

print("import했을 때 __name__:", mypkg.run.__name__)  # "__main__"이 아니므로 print가 실행되지 않았다
print("함수는 쓸 수 있다     :", mypkg.run.shout("imported"))

import했을 때 __name__: mypkg.run
함수는 쓸 수 있다     : IMPORTED!


> **노트북에서 `src`를 불러오려면?** 노트북은 `studys/`에서 실행되므로 `src/`가 보이지 않습니다. 첫 셀에서 `sys.path.insert(0, "..")`처럼 프로젝트 루트를 경로에 추가하면 `from src.tfidf import NumpyTfidf`가 동작합니다.

## 1.4 KoNLPy와 Java

한국어 형태소 분석기 **Okt**는 KoNLPy 패키지에 들어 있습니다. 그런데 KoNLPy는 파이썬 겉면만 있고, Okt의 실제 분석 엔진은 **Java(Scala)로 작성**되어 있습니다. KoNLPy는 `JPype`라는 도구로 파이썬 안에서 **Java 가상 머신(JVM)**을 띄워 그 엔진을 호출합니다.

```
파이썬 코드 ── konlpy ── JPype ── JVM(Java) ── Okt 분석 엔진
```

그래서 `uv add konlpy`만으로는 부족하고 **JDK(Java 개발 도구)**가 따로 설치되어 있어야 합니다.

```bash
# macOS
brew install openjdk@17
# 설치 후 brew가 안내하는 symlink 명령을 실행하거나 JAVA_HOME을 설정
export JAVA_HOME=$(/usr/libexec/java_home)

# Ubuntu
sudo apt install openjdk-17-jdk

# Windows: Temurin JDK 17을 설치하고 환경 변수 JAVA_HOME을 JDK 폴더로 지정
```

| 에러 메시지 | 원인 | 해결 |
|---|---|---|
| `JVMNotFoundException` | Java가 없거나 위치를 못 찾음 | JDK 설치, `JAVA_HOME` 설정 |
| `No JVM shared library file (libjvm...)` | `JAVA_HOME`이 JDK가 아닌 엉뚱한 폴더 | `JAVA_HOME`을 JDK 폴더로 수정 |
| Apple Silicon에서 아키텍처 오류 | x86용 JDK와 arm64 파이썬이 섞임 | arm64용 JDK 설치 |

아래 셀로 Java와 Okt가 동작하는지 확인합니다. **첫 실행은 JVM을 띄우느라 몇 초 걸립니다.**

In [6]:
import shutil

java = shutil.which("java")
if java is None:
    print("❌ java 명령을 찾을 수 없습니다. JDK를 설치하세요.")
else:
    out = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print("✅ Java:", out.stderr.splitlines()[0])  # java -version은 stderr로 출력한다

try:
    from konlpy.tag import Okt

    okt = Okt()
    sentence = "학교에 갔다가 도서관에서 책을 읽었다"
    print("✅ Okt 동작")
    print("공백 토큰화:", sentence.split())
    print("형태소 분석:", okt.morphs(sentence))
except Exception as e:  # JVM 관련 에러는 종류가 다양해서 넓게 잡는다
    print("❌ Okt 실행 실패:", type(e).__name__, e)

✅ Java: openjdk version "11.0.27" 2025-04-15


✅ Okt 동작
공백 토큰화: ['학교에', '갔다가', '도서관에서', '책을', '읽었다']


형태소 분석: ['학교', '에', '갔다가', '도서관', '에서', '책', '을', '읽었다']


공백으로 자르면 `학교에`, `도서관에서`처럼 조사가 붙은 덩어리가 나오지만, 형태소 분석은 `학교 / 에`, `도서관 / 에서`처럼 나눕니다. 이 차이가 검색과 분류 성능에 어떤 영향을 주는지는 9장에서 실험합니다.

## 1.5 NLTK 데이터

NLTK는 **라이브러리**와 **데이터**(불용어 목록, 토크나이저 모델 등)가 따로 배포됩니다. 라이브러리를 설치한 뒤 필요한 데이터를 한 번 내려받아야 하며, 데이터는 홈 폴더의 `nltk_data/`에 저장됩니다.

| 데이터 | 쓰는 곳 |
|---|---|
| `stopwords` | 영어 불용어 목록 (10장) |
| `punkt_tab` | `word_tokenize` 문장·단어 분리 모델 (9장) |

아래 함수는 데이터가 이미 있으면 그대로 두고, 없을 때만 내려받습니다.

In [7]:
import nltk


def ensure_nltk(resource: str, package: str) -> str:
    try:
        nltk.data.find(resource)
        return "이미 있음"
    except LookupError:
        return "다운로드함" if nltk.download(package, quiet=True) else "❌ 다운로드 실패"


for resource, package in [("corpora/stopwords", "stopwords"), ("tokenizers/punkt_tab", "punkt_tab")]:
    print(f"{package:<10} {ensure_nltk(resource, package)}")

stopwords  이미 있음
punkt_tab  이미 있음


In [8]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

en_stop = stopwords.words("english")
print("영어 불용어 수:", len(en_stop))
print("앞 15개      :", en_stop[:15])

text = "NASA's new rocket didn't launch; it costs $1.5B."
print("\n공백 split    :", text.split())
print("word_tokenize :", word_tokenize(text))

영어 불용어 수: 198
앞 15개      : ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]

공백 split    : ["NASA's", 'new', 'rocket', "didn't", 'launch;', 'it', 'costs', '$1.5B.']
word_tokenize : ['NASA', "'s", 'new', 'rocket', 'did', "n't", 'launch', ';', 'it', 'costs', '$', '1.5B', '.']


`split()`은 `launch;`처럼 문장부호가 붙은 채로 자르지만, `word_tokenize`는 `NASA` + `'s`, `did` + `n't`처럼 영어 규칙에 맞춰 나눕니다. 어느 쪽이 이 과제에 맞는지는 9장에서 정합니다.

## 1.6 requirements.txt와 .gitignore

### requirements.txt
평가자는 제출된 저장소를 받아 `pip install -r requirements.txt`로 환경을 만들고 코드를 실행합니다. 그래서 **이 과제가 실제로 쓰는 라이브러리와 버전**을 적어야 합니다.

| 표기 | 의미 |
|---|---|
| `numpy==2.5.3` | 정확히 이 버전 (재현성이 가장 좋음) |
| `numpy>=2.0` | 이 버전 이상이면 아무거나 |

만드는 방법은 두 가지입니다.
1. `uv export --format requirements-txt --no-hashes > requirements.txt`: 환경 전체를 자동으로 적습니다. 하지만 이 저장소처럼 **여러 과제가 환경 하나를 공유하면** 이 과제와 상관없는 패키지까지 수십 개 들어갑니다.
2. **직접 작성**: 이 과제에서 import하는 라이브러리만 골라 현재 설치된 버전으로 적습니다. 이 과제에서는 이 방법을 씁니다.

이 과제 폴더의 `requirements.txt`는 다음과 같습니다. 줄마다 왜 필요한지 주석을 달아 두면 리뷰하는 사람이 이해하기 쉽습니다.

In [9]:
print((PROJECT / "requirements.txt").read_text(encoding="utf-8"))

# A4-1 TF-IDF 문서 검색·분류 (Python 3.13 에서 확인, 3.10 이상)
# 핵심 계산 (TF-IDF, 코사인 유사도)
numpy==2.5.3
# 데이터 처리
pandas==3.0.6
# 분류 모델·평가, TF-IDF 검증, 20 Newsgroups 로드
scikit-learn==1.9.1
# 혼동 행렬 그림
matplotlib==3.11.2
# 영어 불용어
nltk==3.10.3
# 한국어 형태소 분석 (Okt, Java 8 이상 필요)
konlpy==0.6.0
# 테스트
pytest==9.1.1



### .gitignore
git에 올리지 않을 파일을 적습니다.

```gitignore
# 원본 데이터와 캐시: 용량이 크고, 라이선스상 재배포가 애매하고, 코드로 다시 받을 수 있다
projects/a4-1-document-search/data/raw/
projects/a4-1-document-search/data/cache/
# 파이썬과 주피터가 자동으로 만드는 파일
__pycache__/
.ipynb_checkpoints/
```

데이터를 올리지 않는 대신, README에 **데이터를 어디서 어떻게 받는지** 반드시 적어야 평가자가 재현할 수 있습니다.

## ✍️ 실습: requirements 줄 만들기

패키지 이름 리스트를 받아, 현재 설치된 버전으로 `이름==버전` 줄을 만들어 하나의 문자열로 돌려주는 함수 `pin(packages)`를 작성하세요. 설치되지 않은 패키지는 `# 이름: 설치 안 됨` 줄로 표시합니다.

```python
print(pin(["numpy", "pandas", "not-a-real-package"]))
# numpy==2.x.x
# pandas==x.x.x
# # not-a-real-package: 설치 안 됨
```

In [10]:
# 여기에 작성하세요

**정답**

In [11]:
def pin(packages: list[str]) -> str:
    lines = []
    for name in packages:
        try:
            lines.append(f"{name}=={version(name)}")
        except PackageNotFoundError:
            lines.append(f"# {name}: 설치 안 됨")
    return "\n".join(lines)


print(pin(["numpy", "pandas", "not-a-real-package"]))

numpy==2.5.3
pandas==3.0.6
# not-a-real-package: 설치 안 됨


## ✅ 환경 점검

아래 셀의 항목이 **모두 ✅**여야 다음 장으로 넘어갈 수 있습니다. ❌가 있으면 해당 절(1.1, 1.4, 1.5)로 돌아가세요.

In [12]:
import importlib

checks = [("Python 3.10 이상", sys.version_info >= (3, 10))]

for module in ["numpy", "pandas", "sklearn", "matplotlib", "seaborn", "nltk", "konlpy", "pytest"]:
    try:
        importlib.import_module(module)
        checks.append((f"import {module}", True))
    except ImportError:
        checks.append((f"import {module}", False))

checks.append(("java 명령", shutil.which("java") is not None))

try:
    from konlpy.tag import Okt

    checks.append(("Okt 형태소 분석", len(Okt().morphs("테스트 문장입니다")) > 0))
except Exception:
    checks.append(("Okt 형태소 분석", False))

for resource in ["corpora/stopwords", "tokenizers/punkt_tab"]:
    try:
        nltk.data.find(resource)
        checks.append((f"NLTK {resource}", True))
    except LookupError:
        checks.append((f"NLTK {resource}", False))

for name, ok in checks:
    print(("✅" if ok else "❌"), name)
print("\n모두 통과!" if all(ok for _, ok in checks) else "\n❌ 항목을 해결하세요.")

✅ Python 3.10 이상
✅ import numpy
✅ import pandas
✅ import sklearn
✅ import matplotlib
✅ import seaborn
✅ import nltk
✅ import konlpy
✅ import pytest
✅ java 명령
✅ Okt 형태소 분석
✅ NLTK corpora/stopwords
✅ NLTK tokenizers/punkt_tab

모두 통과!


## 🧐 퀴즈

**Q1.** 프로젝트 폴더에서 `python src/verify.py`를 실행했더니 `ModuleNotFoundError: No module named 'src'`가 났습니다. 이유와 해결 방법은?
<details><summary>▶ 정답 보기</summary>

파일 경로로 실행하면 `sys.path`의 첫 위치가 그 파일이 있는 `src/` 폴더가 됩니다. `src/` 안에는 `src`라는 폴더가 없으니 `from src.x import ...`가 실패합니다. 프로젝트 폴더에서 `python -m src.verify`로 실행하면 현재 폴더(프로젝트 루트)가 경로에 들어가서 해결됩니다.
</details>

**Q2.** `uv add konlpy`로 설치까지 성공했는데 `Okt()`에서 JVM 에러가 납니다. 왜일까요?
<details><summary>▶ 정답 보기</summary>

KoNLPy는 파이썬 래퍼일 뿐이고, Okt 분석 엔진은 Java로 만들어져 있어서 JVM을 띄워야 합니다. JDK가 없거나 `JAVA_HOME`이 잘못되어 있으면 파이썬 패키지 설치와 상관없이 실패합니다.
</details>

**Q3.** 원본 데이터를 git에 올리지 않는 이유를 두 가지 이상 말해 보세요. 대신 무엇을 해야 할까요?
<details><summary>▶ 정답 보기</summary>

용량이 크고, 데이터셋 라이선스상 재배포가 제한될 수 있고, 코드로 다시 받을 수 있기 때문입니다. 대신 README에 데이터 출처와 받는 방법, 놓을 위치를 적어서 평가자가 재현할 수 있게 해야 합니다.
</details>

**Q4.** `uv export`로 만든 `requirements.txt` 대신 직접 작성하는 이유는 무엇일까요?
<details><summary>▶ 정답 보기</summary>

이 저장소는 여러 과제가 환경 하나를 공유해서, `uv export`를 쓰면 이 과제가 쓰지 않는 패키지까지 모두 들어가기 때문입니다. 과제가 실제로 import하는 라이브러리만 적어야 의존성이 분명해집니다.
</details>

## ✅ 체크포인트
- [ ] 환경 점검 셀이 모두 ✅이다.
- [ ] `uv add`, `uv run`이 각각 무엇을 하는지 말할 수 있다.
- [ ] `python -m`으로 실행해야 하는 이유를 설명할 수 있다.
- [ ] `src/`, `tests/`, `outputs/`, `data/` 폴더의 역할을 안다.

## 🔗 과제 연결과 다음 장
- 과제 요구사항 "실행 환경: Python 3.10+, `requirements.txt` 제출 필수"를 이 장에서 준비했습니다.
- **2장. 이 과제에 필요한 파이썬**에서 단어 세기, 정렬, 클래스 등 앞으로 계속 쓸 문법을 익힙니다.